# 🚁 Notebook 09 — Practical Control Issues

### When the real world fights back

Every controller so far flew in an idealized world: a perfect sensor, unlimited motors, no wind, no
delay. Real drones don't. This notebook deliberately **breaks those assumptions** — one at a time —
so you can see how each imperfection degrades your beautifully tuned PID, and how to defend against
it. This is the difference between a controller that works in a notebook and one that works on
hardware.

The big word for "keeps working despite imperfections and disturbances" is **robustness**, and
that's what we're building intuition for.

---

## 🎯 Learning objectives
- See how **sensor noise**, **actuator saturation**, **wind**, and **measurement delay** each hurt a PID.
- Apply the right defense to each (filtering, anti-windup, integral action, and slowing the loop).
- Combine all disturbances into a stress test and tune for **robustness**.

## 🧩 Prerequisites
Notebooks 01–08 (the full PID controller, tuning, metrics).

## ⏱️ Estimated time
About **50–65 minutes**.

## 🧠 Concepts you know so far
Full PID · tuning · metrics · anti-windup · derivative filtering · the engine's noise/wind/delay/limit switches

## 🔜 Concepts you'll learn here
Sensor noise · actuator saturation · wind disturbance · measurement delay · robustness


### 🔁 Quick recap
Our `simulate()` engine already has the switches we need: `noise_std`, `wind`, `delay_steps`, and
`max_thrust`/`min_thrust`. We'll flip them on, one at a time, and watch what happens. Run setup and
load the engine + controller + metrics helper.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80
plt.rcParams["figure.figsize"] = (8, 4)

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready!")


### 🧰 Engine + PID controller + metrics (from Notebooks 07–08)

In [ ]:
from collections import deque

def simulate(controller, target=10.0, mass=1.0, g=9.8, start_height=0.0,
             dt=0.02, total_time=12.0, max_thrust=30.0, min_thrust=0.0,
             noise_std=0.0, wind=0.0, delay_steps=0, seed=0):
    """Run the 1D drone with ANY controller and return recorded signals as arrays.

    `controller` is a callable: controller(target, measured_altitude, dt) -> desired_thrust.
    It may hold internal state (integral, previous error, etc.) and may expose a dict
    `controller.last_terms = {'p':..,'i':..,'d':..}` which we record for the dashboards.

    Physical extras (all optional, default off):
      noise_std  : Gaussian sensor noise standard deviation (metres)
      wind       : constant extra force in newtons (+ up, - down)
      delay_steps: how many steps the sensor reading lags behind reality
      max/min_thrust : actuator saturation limits (newtons)
    """
    rng = np.random.default_rng(seed)
    x, v, t = start_height, 0.0, 0.0
    buf = deque([start_height] * (delay_steps + 1), maxlen=delay_steps + 1)  # sensor delay line

    keys = ("t", "x", "v", "a", "measured", "thrust", "error", "p", "i", "d")
    hist = {k: [] for k in keys}

    for _ in range(int(total_time / dt)):
        buf.append(x)                                    # newest true altitude enters the line
        delayed = buf[0]                                 # controller sees the OLD reading
        measured = delayed + (rng.normal(0, noise_std) if noise_std > 0 else 0.0)
        error = target - measured

        thrust = controller(target, measured, dt)        # <-- the controller decides
        thrust = min(max(thrust, min_thrust), max_thrust)  # actuator can't exceed limits

        net_force = thrust - mass * g + wind             # sum of forces (up +, down -)
        a = net_force / mass                             # Newton's second law

        terms = getattr(controller, "last_terms", {"p": 0.0, "i": 0.0, "d": 0.0})
        for k, val in zip(keys, (t, x, v, a, measured, thrust, error,
                                 terms.get("p", 0.0), terms.get("i", 0.0), terms.get("d", 0.0))):
            hist[k].append(val)

        v = v + a * dt                                   # Euler integrate
        x = x + v * dt
        if x < 0:                                        # ground
            x, v = 0.0, 0.0
        t = t + dt

    return {k: np.array(val) for k, val in hist.items()}


In [ ]:
class PIDController:
    """A complete PID controller with anti-windup and a filtered derivative.

    control law:  thrust = Kp*error + Ki*(integral of error) + Kd*(derivative)
    - NO gravity feed-forward on purpose, so the INTEGRAL term is what learns to
      cancel gravity. This is why removing integral leaves a steady-state error.
    - derivative is taken on the MEASUREMENT (not the error) and low-pass filtered,
      which avoids 'derivative kick' and tames sensor noise.
    """
    def __init__(self, Kp=0.0, Ki=0.0, Kd=0.0,
                 i_limit=15.0, d_filter=0.1, use_anti_windup=True):
        self.Kp, self.Ki, self.Kd = Kp, Ki, Kd
        self.i_limit = i_limit            # clamp on the integral term (anti-windup)
        self.d_filter = d_filter          # 1.0 = no smoothing, small = heavy smoothing
        self.use_anti_windup = use_anti_windup
        self.integral = 0.0
        self.prev_measured = None
        self.d_state = 0.0
        self.last_terms = {"p": 0.0, "i": 0.0, "d": 0.0}

    def __call__(self, target, measured, dt):
        error = target - measured

        # --- Proportional: react to the error RIGHT NOW ---
        p = self.Kp * error

        # --- Integral: accumulate error over time (with anti-windup clamp) ---
        self.integral += error * dt
        if self.use_anti_windup and self.Ki > 0:
            hi = self.i_limit / self.Ki
            self.integral = max(-hi, min(hi, self.integral))   # clamp integral
        i = self.Ki * self.integral

        # --- Derivative (on measurement, filtered): anticipate the future ---
        if self.prev_measured is None:
            raw = 0.0
        else:
            raw = (measured - self.prev_measured) / dt          # rate of change of altitude
        self.prev_measured = measured
        self.d_state += self.d_filter * (raw - self.d_state)    # low-pass filter
        d = -self.Kd * self.d_state                             # minus: opposes motion (damping)

        self.last_terms = {"p": p, "i": i, "d": d}
        return p + i + d


In [ ]:
def performance_metrics(t, x, target, band=0.05):
    """Compute rise time, overshoot, settling time and steady-state error from a response."""
    t = np.asarray(t, float); x = np.asarray(x, float)
    final = float(np.mean(x[int(0.9 * len(x)):]))          # average of last 10% = steady state
    sse = target - final                                   # steady-state error (signed)

    thresh = 0.9 * target                                  # rise time = first reach 90% of target
    idx = np.where(x >= thresh)[0]
    rise = float(t[idx[0]]) if len(idx) else float("nan")

    peak = float(np.max(x))                                # overshoot = how far past target, in %
    overshoot = max(0.0, (peak - target) / target * 100.0)

    tol = band * target                                    # settling = last exit from +/-5% band
    outside = np.where(np.abs(x - target) > tol)[0]
    settle = float(t[outside[-1]]) if len(outside) else 0.0

    return dict(rise_time=rise, overshoot_pct=overshoot,
                settling_time=settle, steady_state_error=sse, final_value=final)


## 0. Our baseline (the ideal world)

Here's the well-tuned PID from Notebook 08 in perfect conditions — our reference to compare against.


In [ ]:
GAINS = dict(Kp=4.0, Ki=1.2, Kd=4.0)
res_ideal = simulate(PIDController(**GAINS), total_time=16.0)
plt.figure(figsize=(9, 4.2))
plt.plot(res_ideal["t"], res_ideal["x"], color="tab:blue", lw=2)
plt.axhline(10, color="seagreen", ls="--"); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Baseline: well-tuned PID, perfect world"); plt.grid(True, alpha=0.3); plt.show()


## 1. Sensor noise 🌫️

Real altitude sensors jitter. Noise mostly hurts through the **D** term (Notebook 06). Defense:
**filter the derivative** (our class already does, via `d_filter`). Let's compare heavy filtering
vs almost none, on a noisy sensor.


In [ ]:
res_nofilt = simulate(PIDController(**GAINS, d_filter=1.0),  total_time=16.0, noise_std=0.06, seed=3)
res_filt   = simulate(PIDController(**GAINS, d_filter=0.08), total_time=16.0, noise_std=0.06, seed=3)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
a1.plot(res_nofilt["t"], res_nofilt["thrust"], color="tab:red",   lw=0.7, label="d_filter=1.0 (none)")
a1.plot(res_filt["t"],   res_filt["thrust"],   color="tab:green", lw=1.4, label="d_filter=0.08 (heavy)")
a1.set_title("Thrust command under sensor noise"); a1.set_ylabel("N"); a1.set_xlabel("s")
a1.legend(); a1.grid(True, alpha=0.3)
a2.plot(res_filt["t"], res_filt["x"], color="tab:blue", lw=1.4); a2.axhline(10, color="seagreen", ls="--")
a2.set_title("Altitude (with filtered D) stays clean"); a2.set_ylabel("m"); a2.set_xlabel("s"); a2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Filtering the derivative turns a violent, motor-hammering thrust into a smooth one.")


## 2. Actuator saturation ⛔

Motors have a maximum thrust. When the demand exceeds it, the command is **clipped** — and, as you
saw in Notebook 05, a clipped integral can **wind up**. Defense: **anti-windup** (already in our
class; toggle with `use_anti_windup`). Watch a demanding climb with tight limits, with and without it.


In [ ]:
res_windup = simulate(PIDController(**GAINS, use_anti_windup=False), target=10.0,
                      max_thrust=12.5, total_time=26.0)
res_ok     = simulate(PIDController(**GAINS, use_anti_windup=True),  target=10.0,
                      max_thrust=12.5, total_time=26.0)
plt.figure(figsize=(9, 4.6))
plt.plot(res_windup["t"], res_windup["x"], color="tab:red",   lw=2, label="no anti-windup (overshoots)")
plt.plot(res_ok["t"],     res_ok["x"],     color="tab:green", lw=2, label="anti-windup (clean)")
plt.axhline(10, color="seagreen", ls="--"); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Saturated motors (max 12.5 N): anti-windup prevents the overshoot")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()


## 3. Wind disturbance 💨

A steady wind is an extra constant force. The good news: the **integral term absorbs constant
disturbances** with zero steady-state error (Notebook 05). Watch a steady downdraft get soaked up,
and see what a controller with `Ki=0` does instead.


In [ ]:
res_I   = simulate(PIDController(Kp=4.0, Ki=1.2, Kd=4.0), total_time=20.0, wind=-4.0)
res_noI = simulate(PIDController(Kp=4.0, Ki=0.0, Kd=4.0), total_time=20.0, wind=-4.0)
plt.figure(figsize=(9, 4.6))
plt.plot(res_I["t"],   res_I["x"],   color="tab:green", lw=2, label="with I (holds target)")
plt.plot(res_noI["t"], res_noI["x"], color="tab:red",   lw=2, label="no I (sags below)")
plt.axhline(10, color="seagreen", ls="--"); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Steady 4 N downdraft: the integral term rejects it; without I the drone sags")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
print("The integral quietly grows to supply the extra thrust the wind demands -> zero steady error.")


## 4. Measurement delay ⏳

Sensors and computation take time, so the controller reacts to a **slightly old** reading. Delay is
insidious: it makes the loop act "blind" for a moment, so aggressive gains that were fine now
**oscillate or go unstable**. Defense: **reduce the gains / slow the loop down** to give the delay
room. Watch increasing delay destabilize a fixed tuning.


In [ ]:
plt.figure(figsize=(9, 5))
for delay_steps, colour in [(0, "tab:blue"), (5, "tab:green"), (12, "tab:orange"), (20, "tab:red")]:
    res = simulate(PIDController(Kp=5.0, Ki=1.0, Kd=5.0), total_time=16.0, delay_steps=delay_steps)
    plt.plot(res["t"], res["x"], lw=1.8, color=colour,
             label=f"delay = {delay_steps} steps ({delay_steps*0.02:.2f}s)")
plt.axhline(10, color="black", ls="--"); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Same gains, growing delay: from clean -> oscillating -> unstable")
plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0, 20); plt.show()
print("More delay = the drone reacts to stale info = it overcorrects. The fix is gentler gains.")


## 5. The full stress test — everything at once

Now we throw **noise + saturation + wind + delay** at the drone together. An aggressive tuning
falls apart; a **robust** tuning (gentler gains, good filtering, anti-windup, enough I to fight the
wind) survives. This is what tuning for the real world means.


In [ ]:
harsh = dict(noise_std=0.05, wind=-3.0, delay_steps=8, max_thrust=16.0, total_time=22.0)

aggressive = PIDController(Kp=9.0, Ki=2.5, Kd=2.0, d_filter=0.5)     # twitchy
robust     = PIDController(Kp=3.5, Ki=1.0, Kd=4.5, d_filter=0.08)    # calm + well-defended

res_agg = simulate(aggressive, **harsh)
res_rob = simulate(robust,     **harsh)

plt.figure(figsize=(9, 5))
plt.plot(res_agg["t"], res_agg["x"], color="tab:red",   lw=1.4, label="aggressive tuning (falls apart)")
plt.plot(res_rob["t"], res_rob["x"], color="tab:green", lw=2,   label="robust tuning (survives)")
plt.axhline(10, color="seagreen", ls="--"); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Noise + wind + delay + saturation, all at once")
plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0, 18); plt.show()

for name, res in [("aggressive", res_agg), ("robust", res_rob)]:
    m = performance_metrics(res["t"], res["x"], 10.0)
    print(f"{name:11}: overshoot {m['overshoot_pct']:5.0f}%  settle {m['settling_time']:5.1f}s  sse {m['steady_state_error']:+.2f} m")


## 6. 🎬 The robust drone surviving the storm


In [ ]:
res = simulate(PIDController(Kp=3.5, Ki=1.0, Kd=4.5, d_filter=0.08),
               noise_std=0.05, wind=-3.0, delay_steps=8, max_thrust=16.0, total_time=18.0)
t, x = res["t"], res["x"]
fids = np.linspace(0, len(t)-1, 120).astype(int)
fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1); ax.set_ylim(0, 13); ax.set_xticks([]); ax.set_ylabel("altitude (m)")
ax.set_title("Robust PID vs a nasty world"); ax.axhline(0, color="saddlebrown", lw=3)
ax.axhline(10, color="seagreen", ls="--", lw=2)
drone, = ax.plot([], [], "o", color="tab:green", markersize=26)
label = ax.text(-0.9, 12, "", fontsize=11)
def init(): drone.set_data([], []); label.set_text(""); return drone, label
def update(f):
    i = fids[f]; drone.set_data([0], [x[i]]); label.set_text(f"t={t[i]:.1f}s\nx={x[i]:.2f} m"); return drone, label
ani = animation.FuncAnimation(fig, update, frames=len(fids), init_func=init, blit=True, interval=45)
plt.close(fig); HTML(ani.to_jshtml())


## 7. 🎛️ Break it yourself — all disturbances live

Push each disturbance up and watch the drone struggle, then re-tune the gains to recover. This is
the single most useful cell for building real-world intuition: there's no perfect tuning, only a
good compromise for the disturbances you actually face.


In [ ]:
def stress(Kp=3.5, Ki=1.0, Kd=4.5, noise=0.05, wind=-3.0, delay_steps=8, max_thrust=16.0):
    ctrl = PIDController(Kp, Ki, Kd, d_filter=0.08)
    res = simulate(ctrl, noise_std=noise, wind=wind, delay_steps=int(delay_steps),
                   max_thrust=max_thrust, total_time=22.0)
    m = performance_metrics(res["t"], res["x"], 10.0)
    plt.figure(figsize=(9, 4.6))
    plt.plot(res["t"], res["x"], color="tab:blue", lw=1.6); plt.axhline(10, color="seagreen", ls="--")
    plt.ylim(0, 18); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
    plt.title(f"overshoot={m['overshoot_pct']:.0f}%  settle={m['settling_time']:.1f}s  sse={m['steady_state_error']:+.2f} m")
    plt.grid(True, alpha=0.3); plt.show()

interact(stress,
         Kp=FloatSlider(min=0.5, max=10.0, step=0.5, value=3.5),
         Ki=FloatSlider(min=0.0, max=4.0,  step=0.2, value=1.0),
         Kd=FloatSlider(min=0.0, max=10.0, step=0.5, value=4.5),
         noise=FloatSlider(min=0.0, max=0.12, step=0.01, value=0.05),
         wind=FloatSlider(min=-6.0, max=6.0, step=0.5, value=-3.0),
         delay_steps=IntSlider(min=0, max=25, step=1, value=8),
         max_thrust=FloatSlider(min=11.0, max=30.0, step=1.0, value=16.0));


## ✅ Summary
- **Sensor noise** → mostly hurts D → **filter the derivative**.
- **Actuator saturation** → causes integral windup → **anti-windup (clamp the integral)**.
- **Steady wind** → an added constant force → **the integral term rejects it** with zero steady error.
- **Measurement delay** → makes aggressive gains unstable → **use gentler gains / slow the loop**.
- **Robustness** = performing acceptably across all of these at once, which means accepting a
  compromise tuning rather than a knife-edge "perfect" one.


## ⚠️ Common mistakes
- **Tuning aggressively on an ideal sim.** Those gains often oscillate the moment delay or noise appears.
- **Blaming the gains for a windup overshoot.** It's the saturation + missing anti-windup, not Kp.
- **Adding huge Kd to fight noise.** That makes noise *worse*; filter instead.
- **Forgetting that delay is destabilizing.** If it oscillates only with delay on, lower the gains.

## 🧭 Mental model
> *"The ideal controller is a myth. Real control is choosing a tuning that's good enough across the
> noise, limits, wind, and delay you'll actually meet — trading a little speed for a lot of safety."*

## 🌍 Real-world applications
Outdoor drones in gusty wind, cheap noisy sensors, networked control with communication lag,
undersized actuators — every field system copes with exactly these four gremlins.


## 🧪 Exercises

**L1 — Observe.** In the delay sweep, at roughly how many delay steps does the fixed tuning start
to seriously oscillate?
<details><summary>▸ Solution</summary>
Around <b>12 steps (~0.24 s)</b> it rings badly, and by 20 steps it's near-unstable. Delay steadily
erodes stability for a fixed set of gains.
</details>

**L2 — Modify.** In the stress test, turn `wind` to a strong updraft (+5). Does the integral still
hold target? How long does it take?
<details><summary>▸ Solution</summary>
Yes — the integral simply settles to a smaller value (less thrust needed with an updraft) and holds
target, though it takes a little while to re-balance after the change.
</details>

**L3 — Predict.** With heavy noise, you *increase* Kd. Predict the effect on the thrust command,
then test with the slider.
<details><summary>▸ Solution</summary>
The thrust gets <b>noisier</b>, because D scales the (noisy) derivative. To use more Kd under noise
you must filter harder (smaller <code>d_filter</code>). More damping and more noise are in tension.
</details>


## ❓ Quick quiz
**Q1.** Which disturbance does the integral term handle beautifully?
<details><summary>▸ Answer</summary>A **steady (constant) disturbance** like a persistent wind — it rejects it with zero steady-state error.</details>

**Q2.** Anti-windup defends against problems caused by…
<details><summary>▸ Answer</summary>**Actuator saturation** (motors hitting their max/min thrust).</details>

**Q3.** Measurement delay tends to make a loop…
<details><summary>▸ Answer</summary>**Less stable** — it oscillates or even goes unstable, because the controller reacts to stale information. The fix is gentler gains.</details>

**Q4.** The right defense against noise reaching the D term is…
<details><summary>▸ Answer</summary>**Low-pass filtering** the derivative (small <code>d_filter</code>) — not adding more Kd.</details>


## 🔭 Preview of Notebook 10 — *Final Drone Project*

Time to put it all together. Notebook 10 is a polished, self-contained **drone flight simulator**: a
professional multi-panel dashboard, an animated drone, automatic performance metrics, and a full
interactive cockpit with every gain and disturbance at your fingertips — plus a set of **challenge
problems** (target changes, a payload drop, gust rejection, minimal-overshoot tuning) to test
everything you've learned. 🚁🎓
